# 베이스라인: Qwen2.5-3B-Instruct 그대로 추론

**목적**: 파인튜닝 없이 베이스 모델만으로 리더보드 문제 1,000개를 풀어 기준 점수를 확보한다.

## Kaggle에서 실행하는 법
1. 대회 페이지 → **Code** 탭 → **New Notebook**
2. 이 노트북 내용을 붙여넣거나 **File → Import Notebook**으로 업로드
3. 오른쪽 설정(Settings)에서:
   - **Accelerator**: GPU T4 x2
   - **Internet**: On (모델 다운로드에 필요)
4. 위에서부터 순서대로 실행 (전체 약 2~3시간 소요)
5. 완료되면 `/kaggle/working/submission.csv`가 생성됨 → 대회 페이지에서 제출

In [ ]:
# ── 1. 데이터 불러오기 ──
# Kaggle에서는 대회 데이터가 /kaggle/input/ 아래에 자동으로 붙는다.
import glob, os
import pandas as pd

def find_csv(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'{name}을 찾지 못했습니다. 오른쪽 Input 패널에서 대회 데이터가 붙어 있는지 확인하세요.'
    return hits[0]

train_df = pd.read_csv(find_csv('deep_chal_math_train.csv'))
lb_df = pd.read_csv(find_csv('deep_chal_math_leaderboard.csv'))
# 컬럼명에 공백이 섞여 있는 경우가 있어 정리 (' answer' → 'answer')
train_df.columns = train_df.columns.str.strip()
lb_df.columns = lb_df.columns.str.strip()

print(f'train {len(train_df)}개, leaderboard {len(lb_df)}개')
print(lb_df.head(2))

In [ ]:
# ── 2. 모델 로드 ──
# fp16(반정밀도)로 로드하면 3B 모델이 약 6GB. T4(16GB) 한 장에 충분히 들어간다.
# device_map='auto'는 GPU가 2장이면 레이어를 나눠서 올려준다.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'left'  # 생성 작업에서는 왼쪽 패딩이 필수

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map='auto',
)
model.eval()
print('모델 로드 완료')

In [ ]:
# ── 3. 프롬프트와 생성 함수 ──
# 모델에게 '단계별로 풀고 최종 답을 \boxed{} 안에 넣어라'고 지시한다.
# 이렇게 하면 나중에 답을 기계적으로 추출하기 쉽다.
from tqdm.auto import tqdm

SYSTEM_PROMPT = (
    'You are an expert competition mathematician. '
    'Solve the problem step by step. '
    'The final answer is always an integer. '
    'Put your final integer answer inside \\boxed{}.'
)

def build_prompt(question):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': question},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

@torch.no_grad()
def generate_batch(questions, max_new_tokens=1024, batch_size=16):
    """질문 리스트를 받아 모델의 풀이 텍스트 리스트를 돌려준다."""
    outputs = []
    for i in tqdm(range(0, len(questions), batch_size)):
        batch = [build_prompt(q) for q in questions[i:i + batch_size]]
        enc = tokenizer(batch, return_tensors='pt', padding=True).to(model.device)
        gen = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # 베이스라인은 항상 같은 답이 나오는 greedy 방식
            pad_token_id=tokenizer.pad_token_id,
        )
        # 입력 부분을 잘라내고 새로 생성된 부분만 디코딩
        new_tokens = gen[:, enc['input_ids'].shape[1]:]
        outputs.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))
    return outputs

In [ ]:
# ── 4. 답 추출 (후처리) ──
# 모델 출력에는 풀이 과정이 섞여 있으므로 최종 정수만 뽑아낸다.
# 우선순위: \boxed{} 안의 값 → 없으면 텍스트의 마지막 정수 → 그래도 없으면 0
import re

def parse_int(s):
    """'1,234', '$-5$', '42.0' 같은 문자열을 정수로 변환. 실패하면 None."""
    s = re.sub(r'[,\s$]', '', s)       # 쉼표, 공백, $ 기호 제거
    s = re.sub(r'\\[!,;:]', '', s)     # \! \, 같은 LaTeX 공백 기호 제거
    s = s.strip('.')
    try:
        return int(s)
    except ValueError:
        pass
    try:
        f = float(s)
        if abs(f - round(f)) < 1e-6:
            return int(round(f))
    except (ValueError, OverflowError):
        pass
    return None

def extract_answer(text):
    # 1순위: \boxed{...} 안의 내용 (마지막 것 사용)
    boxed = re.findall(r'\\boxed\{([^{}]+)\}', text)
    for cand in reversed(boxed):
        val = parse_int(cand)
        if val is not None:
            return val
    # 2순위: 텍스트에 등장하는 마지막 정수
    nums = re.findall(r'-?\d[\d,]*', text)
    for cand in reversed(nums):
        val = parse_int(cand)
        if val is not None:
            return val
    return 0  # 빈 값은 무조건 오답이므로 아무 정수라도 낸다

# 간단 동작 확인
assert extract_answer('... so the answer is \\boxed{-42}.') == -42
assert extract_answer('The total is 1,234 dollars') == 1234
print('답 추출 함수 OK')

In [ ]:
# ── 5. 미니 평가: train 문제 50개로 실력 측정 ──
# 제출 전에 대략적인 정확도를 미리 확인한다 (약 5~10분 소요).
EVAL_N = 50
eval_df = train_df.sample(EVAL_N, random_state=42).reset_index(drop=True)

eval_outputs = generate_batch(eval_df['question'].tolist())
eval_preds = [extract_answer(o) for o in eval_outputs]
correct = sum(int(p) == int(a) for p, a in zip(eval_preds, eval_df['answer']))
print(f'미니 평가 정확도: {correct}/{EVAL_N} = {correct/EVAL_N:.1%}')

# 틀린 문제 하나를 눈으로 확인해 보기
for i in range(EVAL_N):
    if int(eval_preds[i]) != int(eval_df['answer'][i]):
        print('--- 틀린 예시 ---')
        print('문제:', eval_df['question'][i][:300])
        print('정답:', eval_df['answer'][i], '/ 모델 답:', eval_preds[i])
        print('모델 풀이 끝부분:', eval_outputs[i][-300:])
        break

In [ ]:
# ── 6. 리더보드 1,000문제 전체 추론 (약 2시간) ──
lb_outputs = generate_batch(lb_df['question'].tolist())
lb_preds = [extract_answer(o) for o in lb_outputs]
print('추론 완료:', len(lb_preds), '개')

In [ ]:
# ── 7. submission.csv 생성 ──
# 주의: 대회 문서에는 'ID'라고 써 있지만 실제 채점기는 소문자 'id'를 요구한다
submission = pd.DataFrame({'id': lb_df['id'], 'answer': lb_preds})
submission['answer'] = submission['answer'].astype('int64')
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.head())
print('저장 완료 → 대회 페이지에서 이 파일을 제출하세요')